# Embedding2Adapter for SLM RAG

Run the cells from top to bottom. Dataset preparation calls Ollama to create embeddings and can take time. The final cell starts HyperNet training.

In [1]:
from collections import Counter
from pathlib import Path
from pprint import pprint
import sys

PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "pyproject.toml").exists()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Project root could not be found.")

project_root = str(PROJECT_ROOT)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from Experiment.src.Embd2Adapter.Embd2Adapter_pipeline import Embd2AdapterPipeline
from Experiment.src.BaseModel.BaseModel_pipeline import BaseModelPipeline

## 1. Initialize the pipeline

This loads and preprocesses the configured datasets, then loads the base language model.

In [2]:
pipeline = Embd2AdapterPipeline()

Dataset: HotPotQA Initialized!
Dataset: Musique Initialized!
Dataset: TwoWikiMultihopQA Initialized!
Dataset: MultiHopRAG Initialized!
Dataset Constructor Initialized!
VectorStore Initialized


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

HyperNetTrainer Initialized!
(To train the hypernet, first call prepare_datasets() to embed the train datasets.
Then, call train() and the hypernet model training starts!)
Embd2Adapter Pipline Initialized!
Experiment Setup Completed. check the datasets whether it is correct or not, and begin training your model.


## 2. Inspect and validate the datasets

In [ ]:
fields = pipeline.dataset_constructor.OUTPUT_FIELDS

for dataset_name, dataset in pipeline.dataset_constructor.datasets.items():
    print(f"\n=== {dataset_name} ===")
    print(f"train rows: {len(dataset.train_ds['query']):,}")
    print(f"eval rows : {len(dataset.eval_ds['query']):,}")

    for split_name, split in (("train", dataset.train_ds), ("eval", dataset.eval_ds)):
        field_lengths = {field: len(split[field]) for field in fields}
        if len(set(field_lengths.values())) != 1:
            raise ValueError(
                f"Misaligned fields in {dataset_name}/{split_name}: {field_lengths}"
            )

    context_counts = Counter(
        len(gold) + len(distractors)
        for gold, distractors in zip(
            dataset.train_ds["gold_context"],
            dataset.train_ds["distractor"],
        )
    )
    print("train contexts per question:", dict(sorted(context_counts.items())))

print("\nAll dataset fields are aligned.")

In [ ]:
for dataset_name, dataset in pipeline.dataset_constructor.datasets.items():
    sample = {
        "query": dataset.train_ds["query"][0],
        "answer": dataset.train_ds["answer"][0],
        "gold_context": dataset.train_ds["gold_context"][0],
        "distractor": dataset.train_ds["distractor"][0],
    }
    print(f"\n=== Sample: {dataset_name} ===")
    pprint(sample, width=120)

## 3. Build the training dataset

This embeds every gold/distractor context with Ollama and tokenizes the prompts. Progress bars are displayed for both operations.

In [ ]:
pipeline.hypernet_trainer.prepare_datasets()

In [ ]:
train_ds = pipeline.hypernet_trainer.train_ds
embedding_shapes = Counter(
    (len(embedding), len(embedding[0]))
    for embedding in train_ds["embedding"]
)
token_lengths = [len(input_ids) for input_ids in train_ds["input_ids"]]

print(f"training rows       : {len(train_ds):,}")
print("embedding shapes    :", dict(sorted(embedding_shapes.items())))
print(f"token length range  : {min(token_lengths):,} - {max(token_lengths):,}")
print(f"max configured length: {pipeline.hypernet_trainer.info['tokenization']['max_length']:,}")

assert len(train_ds) > 0
assert all(shape[1] == pipeline.embd_model.dim for shape in embedding_shapes)
assert max(token_lengths) <= pipeline.hypernet_trainer.info["tokenization"]["max_length"]
print("Prepared training dataset validation: OK")

## 4. Start HyperNet training

The next cell starts the full training run. Check the summaries above before executing it. Training history is saved through `Recorder` and returned as `log_history`.

In [ ]:
#log_history = pipeline.train_TypeMeanEmbd()
#log_history[-5:]

## 5. Load and evaluate the trained HyperNet

Load the saved HyperNet parameters, then evaluate the complete evaluation split.

In [3]:
checkpoint_path = pipeline.load_trained_hypernet()
checkpoint_path

Loaded trained HyperNet from outputs\hyperformer\hypernet_state_dict.pt


WindowsPath('outputs/hyperformer/hypernet_state_dict.pt')

In [ ]:
pipeline._evaluate(
    "hotpotqa/hotpot_qa",
    use_rag=False,
)

In [4]:
pipeline._evaluate(
    "framolfese/2WikiMultihopQA",
    use_rag=False,
)

Evaluating framolfese/2WikiMultihopQA (No RAG):   0%|          | 0/500 [00:00<?, ?question/s]

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Evaluation Process Initialized
Result appended to C:\Atelier\Research\Embd2Adapter_for_RAG\Experiment\Result\experiment_2026_08_12.json


{'exact_match': 0.7, 'token_f1': 0.6978354756354757}

In [2]:
basemodel_pipeline = BaseModelPipeline()

Dataset: HotPotQA Initialized!
Dataset: Musique Initialized!
Dataset: TwoWikiMultihopQA Initialized!
Dataset: MultiHopRAG Initialized!
Dataset Constructor Initialized!
LLM initialized
VectorStore Initialized


In [3]:
basemodel_pipeline._evaluate("framolfese/2WikiMultihopQA", use_rag=False)

Evaluating framolfese/2WikiMultihopQA (No RAG):   0%|          | 0/500 [00:00<?, ?question/s]

Evaluation Process Initialized
Result appended to C:\Atelier\Research\Embd2Adapter_for_RAG\Experiment\Result\experiment_2026_08_12.json


{'exact_match': 0.286, 'token_f1': 0.2800973137973138}